# 香港特区`CBERS-04`卫星`MUX`影像卫星姿态采样数据空间插值

## 程序包初始化

### 内建程序包

In [1]:
import io, sys, os; 
try:
    nb_dir = nb_dir; 
except NameError: 
    nb_dir = os.getcwd(); 
    sys.path.append(nb_dir); 

In [2]:
import re; 
import collections as coll, itertools as it; 
from __future__ import print_function; 

In [3]:
from datetime import datetime; 

### 第三方和自定义程序包

In [4]:
import numpy as np, pandas as pd; 

In [5]:
_ = sys.stdout; 
with io.BytesIO() as sys.stdout: import idlpy; 
sys.stdout = _; 

In [6]:
py_pkg_dir = os.path.normpath(
    os.path.join(nb_dir, os.pardir, os.pardir, "Python_Package")
); 
sys.path.append(py_pkg_dir); 

In [7]:
import path_matcher; 
import cresda_metadata_parse as cresda; 
cresda = cresda.importlib.reload(cresda); 

In [8]:
idl_pkg_dir = os.path.normpath(
    os.path.join(nb_dir, os.pardir, os.pardir, "IDL_Package")
); 

In [9]:
idl_satangle = os.path.normpath(
    os.path.join(idl_pkg_dir, "CRESDA_L2_Satangle")
); 
idl_satangle_subpkg = (
    "math", 
    "spatial_analysis", 
    "cresda_l2_param"
); 
idl_satangle_cmpl_seq = tuple(
    os.path.join(idl_satangle, pkg + ".pro")
    for pkg in idl_satangle_subpkg
); 

In [10]:
idlpy.IDL.e = idlpy.IDL.envi(headless=False); 

% Restored file: ENVI.
% Loaded DLM: HPGRAPHICS.
% Compiled module: ENVI_VECTOR_MASK_RASTER_CLASSIC.
% Loaded DLM: PNG.
% Loaded DLM: URL.


In [11]:
for pkg in idl_satangle_cmpl_seq: 
    idlpy.IDL.run(".compile {pkg}".format(pkg=pkg)); 

## 影像辅助数据读取

### 影像存放路径定位

In [12]:
rs_meta_dir = os.path.normpath(
    os.path.join(
        nb_dir, os.pardir, os.pardir, 
        os.pardir, "Source", "Imagery"
    )
); 

### 文件名匹配

In [13]:
mux_finder = cresda.cb04.MUX(); 
mux_finder.source_dir = rs_meta_dir; 
mux_finder.target_dir = rs_meta_dir; 

In [14]:
cb04_mux_scenes = tuple(scene for scene in mux_finder.traverse()); 

## 读取`SatAngle.txt`文件, 插值, 输出
插值方法: `IDL`内置`GRIDDATA`函数, 指定`METHOD="Linear"`
* 以样本点为不规则三角网顶点, 对插值区域行`Delaunay`三角部分后, 执行线性插值. 
* 此方法插值结果与`ENVI`工具箱中`Topographic Tools` > `Rasterize Point Data`工具的中的`linear`方法插值结果一致. 

In [15]:
for scene in cb04_mux_scenes: 
    xml_path = scene.target[0]; 
    output_path = scene.groupdict["archive"]; 
    output_path = os.path.join(nb_dir, output_path + "_obsv_geom.dat"); 
    #跳过已经输出的文件, 防止意外覆盖
    if os.path.isfile(output_path): 
        continue; 
    idlpy.IDL.cresda_l2_obsv_geom(
        xml_path, nb_dir, compression=True, no_open=True
    ); 
    #显示进度
    print("Product {arx} processed at {time}".format(
        arx=scene.groupdict["archive"], 
        time=datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    ), end="\r"); 
    sys.stdout.flush(); 
print("{0: <79}".format("All products have been processed. ")); 
sys.stdout.flush(); 

% Loaded DLM: NATIVE.
% Loaded DLM: JPEG2000.
% Loaded DLM: JPEG.
% Loaded DLM: HDF5.
% Loaded DLM: MAP_PE.
% Compiled module: STDEV.
All products have been processed.                                              
